# Ejemplo motor ingesta para datos en streaming - event-hubs

In [1]:
%run ./includes/famia_stream_generator.ipynb

In [2]:
from databricks.sdk.runtime import dbutils, spark

dbutils.library.restartPython()

In [3]:
import sys

sys.path.append("../src")

from motor_ingesta import MotorIngesta, ConfigLoader

In [4]:
from pathlib import Path


config = ConfigLoader.from_file((Path.cwd().parent / 'kafka-config.json'))

In [5]:
namespace = dbutils.secrets.get(scope="lakehouses-ucm", key="namespace")
connection_string = dbutils.secrets.get(scope="lakehouses-ucm", key="connection-string")

farmia_generator = FamiaStreamGenerator(namespace, connection_string)

In [6]:
current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
schema = "default"
landing = "-"
volume_name = "_meta"
meta = f"/Volumes/{current_catalog}/{schema}/{volume_name}"


spark.sql(f"CREATE VOLUME IF NOT EXISTS {current_catalog}.{schema}.{volume_name}")

""


In [7]:
motor_ingesta = MotorIngesta(
    catalog=current_catalog,
    schema=schema,
    landing=landing,
    meta=meta,
)

## Ingesta de mensajes en forma de eventos JSON desde Kafka

### Generación de eventos

In [ ]:
farmia_generator.create_topic("sensors-json")
await farmia_generator.produce_events("json", "sensors-json", 50)

### Consumo de eventos con el motor de ingesta

In [8]:
config_json = config[:1]

print(config_json[0].model_dump_json(indent=2)[:500])

{
  "source": {
    "format": "event-hubs",
    "messages": "json",
    "options": {
      "includeHeaders": "true",
      "startingOffsets": "earliest"
    },
    "connection": {
      "bootstrap_servers": "mde-lakehouses.servicebus.windows.net:9093",
      "security_protocol": "SASL_SSL",
      "sasl_mechanism": "PLAIN",
      "password": {
        "scope": "lakehouses-ucm",
        "key": "connection-string"
      }
    },
    "schema_": "event_id string, event_ts string, session_id string, c


In [9]:
queries = motor_ingesta.ingest(config_json)

for query in queries:
    query.awaitTermination()
    print(f'Query "{query.name}" finalizada correctamente.')

Query "sensors_json_raw" finalizada correctamente.


In [10]:
%sql
SELECT COUNT(*) FROM default.sensors_json_raw;

,count(1)
0,100


In [11]:
%sql
SELECT * FROM default.sensors_json_raw LIMIT 10;

,_key,_value,_topic,_partition,_offset,_timestamp,_timestampType,_headers,_ingested_at,event_id,event_ts,session_id,customer_id,event_type,product_id,channel
0,b'S00002',"b'{""event_id"": ""APP-000001"", ""event_ts"": ""2026-04-03T09:00:15"", ""session_id"": ""S00002"", ""customer_id"": ""C00002"", ""event_type"": ""add_to_cart"", ""product_id"": ""P0002"", ""channel"": ""ios""}'",sensors-json,0,0,2026-09-11 22:40:19.274,0,"[{'key': 'content-type', 'value': b'application/json'}]",2026-09-11 22:47:49.539,APP-000001,2026-04-03T09:00:15,S00002,C00002,add_to_cart,P0002,ios
1,b'S00003',"b'{""event_id"": ""APP-000002"", ""event_ts"": ""2026-04-03T09:00:30"", ""session_id"": ""S00003"", ""customer_id"": ""C00003"", ""event_type"": ""checkout"", ""product_id"": ""P0003"", ""channel"": ""webview""}'",sensors-json,0,1,2026-09-11 22:40:19.399,0,"[{'key': 'content-type', 'value': b'application/json'}]",2026-09-11 22:47:49.539,APP-000002,2026-04-03T09:00:30,S00003,C00003,checkout,P0003,webview
2,b'S00004',"b'{""event_id"": ""APP-000003"", ""event_ts"": ""2026-04-03T09:00:45"", ""session_id"": ""S00004"", ""customer_id"": ""C00004"", ""event_type"": ""session_start"", ""product_id"": ""P0004"", ""channel"": ""webview""}'",sensors-json,0,2,2026-09-11 22:40:19.462,0,"[{'key': 'content-type', 'value': b'application/json'}]",2026-09-11 22:47:49.539,APP-000003,2026-04-03T09:00:45,S00004,C00004,session_start,P0004,webview
3,b'S00005',"b'{""event_id"": ""APP-000004"", ""event_ts"": ""2026-04-03T09:01:00"", ""session_id"": ""S00005"", ""customer_id"": ""C00005"", ""event_type"": ""purchase"", ""product_id"": ""P0005"", ""channel"": ""ios""}'",sensors-json,0,3,2026-09-11 22:40:19.524,0,"[{'key': 'content-type', 'value': b'application/json'}]",2026-09-11 22:47:49.539,APP-000004,2026-04-03T09:01:00,S00005,C00005,purchase,P0005,ios
4,b'S00006',"b'{""event_id"": ""APP-000005"", ""event_ts"": ""2026-04-03T09:01:15"", ""session_id"": ""S00006"", ""customer_id"": ""C00006"", ""event_type"": ""purchase"", ""product_id"": ""P0006"", ""channel"": ""android""}'",sensors-json,0,4,2026-09-11 22:40:19.587,0,"[{'key': 'content-type', 'value': b'application/json'}]",2026-09-11 22:47:49.539,APP-000005,2026-04-03T09:01:15,S00006,C00006,purchase,P0006,android
5,b'S00007',"b'{""event_id"": ""APP-000006"", ""event_ts"": ""2026-04-03T09:01:30"", ""session_id"": ""S00007"", ""customer_id"": ""C00007"", ""event_type"": ""view_product"", ""product_id"": ""P0007"", ""channel"": ""android""}'",sensors-json,0,5,2026-09-11 22:40:19.665,0,"[{'key': 'content-type', 'value': b'application/json'}]",2026-09-11 22:47:49.539,APP-000006,2026-04-03T09:01:30,S00007,C00007,view_product,P0007,android
6,b'S00008',"b'{""event_id"": ""APP-000007"", ""event_ts"": ""2026-04-03T09:01:45"", ""session_id"": ""S00008"", ""customer_id"": ""C00008"", ""event_type"": ""session_start"", ""product_id"": ""P0008"", ""channel"": ""ios""}'",sensors-json,0,6,2026-09-11 22:40:19.734,0,"[{'key': 'content-type', 'value': b'application/json'}]",2026-09-11 22:47:49.539,APP-000007,2026-04-03T09:01:45,S00008,C00008,session_start,P0008,ios
7,b'S00009',"b'{""event_id"": ""APP-000008"", ""event_ts"": ""2026-04-03T09:02:00"", ""session_id"": ""S00009"", ""customer_id"": ""C00009"", ""event_type"": ""checkout"", ""product_id"": ""P0009"", ""channel"": ""webview""}'",sensors-json,0,7,2026-09-11 22:40:19.803,0,"[{'key': 'content-type', 'value': b'application/json'}]",2026-09-11 22:47:49.539,APP-000008,2026-04-03T09:02:00,S00009,C00009,checkout,P0009,webview
8,b'S00010',"b'{""event_id"": ""APP-000009"", ""event_ts"": ""2026-04-03T09:02:15"", ""session_id"": ""S00010"", ""customer_id"": ""C00010"", ""event_type"": ""checkout"", ""product_id"": ""P0010"", ""channel"": ""webview""}'",sensors-json,0,8,2026-09-11 22:40:19.869,0,"[{'key': 'content-type', 'value': b'application/json'}]",2026-09-11 22:47:49.539,APP-000009,2026-04-03T09:02:15,S00010,C00010,checkout,P0010,webview
9,b'S00011',"b'{""event_id"": ""APP-000010"", ""even

## Ingesta de mensajes en forma de eventos AVRO desde Kafka

### Generación de eventos

In [ ]:
farmia_generator.create_topic("sensors-avro")
await farmia_generator.produce_events("avro", "sensors-avro", 50)

### Consumo de eventos con el motor de ingesta

In [12]:
config_avro = config[1:]

print(config_avro[0].model_dump_json(indent=2)[:500])

{
  "source": {
    "format": "event-hubs",
    "messages": "avro",
    "options": {
      "includeHeaders": "true",
      "startingOffsets": "earliest"
    },
    "connection": {
      "bootstrap_servers": "mde-lakehouses.servicebus.windows.net:9093",
      "security_protocol": "SASL_SSL",
      "sasl_mechanism": "PLAIN",
      "password": {
        "scope": "lakehouses-ucm",
        "key": "connection-string"
      }
    },
    "schema_": "{\"type\":\"record\",\"name\":\"AppEvent\",\"fields\":


In [14]:
queries = motor_ingesta.ingest(config_avro)

for query in queries:
    query.awaitTermination()
    print(f'Query "{query.name}" finalizada correctamente.')

Query "sensors_avro_raw" finalizada correctamente.


In [15]:
%sql
SELECT COUNT(*) FROM default.sensors_avro_raw;

,count(1)
0,50


In [16]:
%sql
SELECT * FROM default.sensors_avro_raw LIMIT 10;

,_key,_value,_topic,_partition,_offset,_timestamp,_timestampType,_headers,_ingested_at,event_id,event_ts,session_id,customer_id,event_type,product_id,channel
0,b'S00002',b'\x14APP-000001&2026-04-03T09:00:15\x0cS00002\x0cC00002\x10purchase\nP0002\x0eandroid',sensors-avro,0,0,2026-09-11 22:50:32.325,0,"[{'key': 'content-type', 'value': b'avro/binary'}]",2026-09-11 23:13:09.273,APP-000001,2026-04-03T09:00:15,S00002,C00002,purchase,P0002,android
1,b'S00003',b'\x14APP-000002&2026-04-03T09:00:30\x0cS00003\x0cC00003\x10purchase\nP0003\x0ewebview',sensors-avro,0,1,2026-09-11 22:50:32.434,0,"[{'key': 'content-type', 'value': b'avro/binary'}]",2026-09-11 23:13:09.273,APP-000002,2026-04-03T09:00:30,S00003,C00003,purchase,P0003,webview
2,b'S00004',b'\x14APP-000003&2026-04-03T09:00:45\x0cS00004\x0cC00004\x10checkout\nP0004\x0ewebview',sensors-avro,0,2,2026-09-11 22:50:32.497,0,"[{'key': 'content-type', 'value': b'avro/binary'}]",2026-09-11 23:13:09.273,APP-000003,2026-04-03T09:00:45,S00004,C00004,checkout,P0004,webview
3,b'S00005',b'\x14APP-000004&2026-04-03T09:01:00\x0cS00005\x0cC00005\x10purchase\nP0005\x0ewebview',sensors-avro,0,3,2026-09-11 22:50:32.575,0,"[{'key': 'content-type', 'value': b'avro/binary'}]",2026-09-11 23:13:09.273,APP-000004,2026-04-03T09:01:00,S00005,C00005,purchase,P0005,webview
4,b'S00006',b'\x14APP-000005&2026-04-03T09:01:15\x0cS00006\x0cC00006\x1asession_start\nP0006\x0eandroid',sensors-avro,0,4,2026-09-11 22:50:32.637,0,"[{'key': 'content-type', 'value': b'avro/binary'}]",2026-09-11 23:13:09.273,APP-000005,2026-04-03T09:01:15,S00006,C00006,session_start,P0006,android
5,b'S00007',b'\x14APP-000006&2026-04-03T09:01:30\x0cS00007\x0cC00007\x10checkout\nP0007\x0eandroid',sensors-avro,0,5,2026-09-11 22:50:32.700,0,"[{'key': 'content-type', 'value': b'avro/binary'}]",2026-09-11 23:13:09.273,APP-000006,2026-04-03T09:01:30,S00007,C00007,checkout,P0007,android
6,b'S00008',b'\x14APP-000007&2026-04-03T09:01:45\x0cS00008\x0cC00008\x10checkout\nP0008\x0ewebview',sensors-avro,0,6,2026-09-11 22:50:32.778,0,"[{'key': 'content-type', 'value': b'avro/binary'}]",2026-09-11 23:13:09.273,APP-000007,2026-04-03T09:01:45,S00008,C00008,checkout,P0008,webview
7,b'S00009',b'\x14APP-000008&2026-04-03T09:02:00\x0cS00009\x0cC00009\x16add_to_cart\nP0009\x06ios',sensors-avro,0,7,2026-09-11 22:50:32.841,0,"[{'key': 'content-type', 'value': b'avro/binary'}]",2026-09-11 23:13:09.273,APP-000008,2026-04-03T09:02:00,S00009,C00009,add_to_cart,P0009,ios
8,b'S00010',b'\x14APP-000009&2026-04-03T09:02:15\x0cS00010\x0cC00010\x10checkout\nP0010\x0ewebview',sensors-avro,0,8,2026-09-11 22:50:32.919,0,"[{'key': 'content-type', 'value': b'avro/binary'}]",2026-09-11 23:13:09.273,APP-000009,2026-04-03T09:02:15,S00010,C00010,checkout,P0010,webview
9,b'S00011',b'\x14APP-000010&2026-04-03T09:02:30\x0cS00011\x0cC00011\x18view_product\nP0011\x0eandroid',sensors-avro,0,9,2026-09-11 22:50:32.981,0,"[{'key': 'content-type', 'value': b'avro/binary'}]",2026-09-11 23:13:09.273,APP-000010,2026-04-03T09:02:30,S00011,C00011,view_product,P0011,android
